# Checkpoint 02 — Trajectory Collection

Generate reproducible action-conditioned RGB transitions from MiniGrid. Generated `.npz` data stays in the Kaggle runtime and is not committed to normal Git history.

In [ ]:
!pip install -q minigrid gymnasium pyyaml pandas matplotlib pillow
!pip install -q "git+https://github.com/ashyx12/dl-assignment.git"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from data import collect_trajectories

## 1. Smoke test

Start with two seeds and a short horizon before generating the larger dataset.

In [ ]:
smoke = collect_trajectories([1, 2], max_steps=10)

print('Transitions:', len(smoke))
print('Observation shape:', smoke.observations.shape)
print('Next observation shape:', smoke.next_observations.shape)
print('Actions:', np.unique(smoke.actions))
print('Seeds:', np.unique(smoke.seeds))
print('Rewards:', smoke.rewards[:10])

## 2. Reproducibility check

In [ ]:
a = collect_trajectories([42], max_steps=10)
b = collect_trajectories([42], max_steps=10)

print('Observations identical:', np.array_equal(a.observations, b.observations))
print('Actions identical:', np.array_equal(a.actions, b.actions))
print('Next observations identical:', np.array_equal(a.next_observations, b.next_observations))

## 3. Inspect a transition

In [ ]:
idx = 0
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(smoke.observations[idx])
axes[0].set_title(f'o_t | action={smoke.actions[idx]}')
axes[0].axis('off')
axes[1].imshow(smoke.next_observations[idx])
axes[1].set_title('o_t+1')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 4. Generate the initial dataset

The project specification targets roughly 10,000 transitions initially. One episode per seed with a maximum of 100 steps gives approximately that scale for 100 seeds, subject to early episode termination.

In [ ]:
train_seeds = list(range(1, 101))
dataset = collect_trajectories(train_seeds, max_steps=100)

print('Transitions:', len(dataset))
print('Observation bytes:', dataset.observations.nbytes / 1e6, 'MB')
print('Unique episodes:', len(np.unique(dataset.episode_ids)))
print('Unique seeds:', len(np.unique(dataset.seeds)))

In [ ]:
path = '/kaggle/working/minigrid_train_10k.npz'
dataset.save(path)
print('Saved:', path)